# 06 - Explainability Generation Node
## 1. Pipeline Objective

## 2. Dependency Management
### 2.1 Package Installation

In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets pyarrow pandas


### 2.2 Module Imports

In [ ]:
import os
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

### 2.3 Environment Variables

In [ ]:
PROJECT_ROOT: str = '/content/drive/MyDrive/LLMReliability'
DATA_PROCESSED_DIR: str = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR: str = os.path.join(PROJECT_ROOT, 'models')
RESULTS_DIR: str = os.path.join(PROJECT_ROOT, 'results')

MODEL_ID: str = "Qwen/Qwen3-4B-Instruct-2507"
OUT_PATH: str = os.path.join(DATA_PROCESSED_DIR, 'explainability_subset.jsonl')

os.environ['HF_HOME'] = MODELS_DIR

for d in [DATA_PROCESSED_DIR, MODELS_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

## 3. Resource Allocation
### 3.1 Identity & Access Management (IAM)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    secrets_path = '/content/drive/MyDrive/secrets.json'
    if os.path.exists(secrets_path):
        with open(secrets_path, 'r') as f:
            secrets = json.load(f)
        hf_token = secrets.get("HF_TOKEN")
        if hf_token:
            from huggingface_hub import login
            login(token=hf_token)
            print("[ENVIRONMENT] status=AUTHENTICATED | component='HuggingFace'")
except ImportError:
    print("[ENVIRONMENT] status=LOCAL | component='Google Drive'")

### 3.2 Dataset Loading (Subset)

In [ ]:
os.chdir(PROJECT_ROOT)

df = pd.read_parquet('data/processed/indopref_main.parquet')
subset_df = df.head(5).copy()
print(
    f"[DATASET] shape={df.shape} | "
    f"subset={len(subset_df)} | "
    f"status=LOADED"
)

## 4. Model Loading

In [ ]:
print(f"[MODEL] id='{MODEL_ID}' | status=INITIALIZING")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
print(f"[MODEL] status=LOADED | vram='{torch.cuda.memory_allocated() / 1024**3:.2f} GB'")

## 5. Prompt Engineering

In [ ]:
def format_json_prompt(user_input: str, resp_A: str, resp_B: str) -> str:
    base = "Evaluasi Response A dan Response B. Pilih respons yang lebih baik secara objektif."
    prompt = f"""{base}\n
Anda HARUS mengevaluasi berdasarkan tiga kriteria:
1. Accuracy: Apakah respons memberikan informasi yang benar?
2. Relevance: Apakah respons menjawab instruksi secara langsung?
3. Clarity: Apakah respons mudah dipahami dan tidak bertele-tele?

Keluarkan hasil evaluasi Anda HANYA dalam format JSON valid. 
PENTING: Anda harus memberikan penjelasan (explanation) terlebih dahulu untuk bernalar, baru kemudian menjatuhkan keputusan akhir (decision) di bagian paling bawah JSON!
Gunakan struktur persis seperti berikut tanpa markdown backticks:
{{
  "explanation": {{
    "accuracy": "Penjelasan singkat",
    "relevance": "Penjelasan singkat",
    "clarity": "Penjelasan singkat"
  }},
  "evidence": [
    {{
      "criterion": "accuracy/relevance/clarity",
      "candidate": "A atau B",
      "evidence": "kutipan teks respons",
      "assessment": "supported atau unsupported"
    }}
  ],
  "criteria": {{
    "accuracy": {{ "A": 9, "B": 8 }},
    "relevance": {{ "A": 8, "B": 9 }},
    "clarity": {{ "A": 9, "B": 9 }}
  }},
  "decision": "A"
}}

[INPUT CONTEXT]
Input: {user_input}

Response A:
{resp_A}

Response B:
{resp_B}
"""
    return prompt

## 6. Explainability Compute Engine
### 6.1 Inference Function Definition

In [ ]:
def run_inference(user_input: str, resp_A: str, resp_B: str) -> str:
    import gc
    raw_prompt = format_json_prompt(user_input, resp_A, resp_B)
    messages = [
        {"role": "system", "content": "You are an impartial and expert AI evaluator. Output valid raw JSON without markdown formatting."},
        {"role": "user", "content": raw_prompt}
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    input_len = inputs['input_ids'].shape[1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
        
    generated_tokens = outputs[0][input_len:]
    res = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    # OOM Shield (Production-Grade Memory Management)
    del inputs, outputs, generated_tokens
    torch.cuda.empty_cache()
    gc.collect()
    
    return res

### 6.2 Compute Loop

In [ ]:
results = []
print(
    f"[INFERENCE] samples={len(subset_df)} | "
    f"mode='symmetrical' | "
    f"status=STARTED"
)

for index, row in subset_df.iterrows():
    # Posisi AB
    raw_ab = run_inference(row['input'], row['response_1'], row['response_2'])
    
    # Posisi BA
    raw_ba = run_inference(row['input'], row['response_2'], row['response_1'])

    print(
        f"[INFERENCE] sample_id={row['original_id']} | "
        f"output_ab={len(raw_ab)} chars | "
        f"output_ba={len(raw_ba)} chars"
    )
    
    results.append({
        "sample_id": row["original_id"],
        "prompt": row["input"],
        "cand_1": row["response_1"],
        "cand_2": row["response_2"],
        "human_preference": row.get("chosen_model", "Unknown"),
        "json_output_ab": raw_ab,
        "json_output_ba": raw_ba
    })

with open(OUT_PATH, 'w', encoding='utf-8') as f:
    for res in results:
        f.write(json.dumps(res, ensure_ascii=False) + '\n')

print(f"[INFERENCE] status=COMPLETED | path='{OUT_PATH}'")